# CSP Demo: Complex State Propagator

This notebook demonstrates how to train and evaluate CSP on the three state tracking tasks.

Paper: [State Propagation Also Satisfies](https://arxiv.org/abs/2608.03425)

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import torch
import matplotlib.pyplot as plt
from csp import CSP, generate_parity_data, generate_mod3_data, generate_parenthesis_data, create_dataloaders
from utils import train_model, evaluate_model, evaluate_model_f1, setup_logging, plot_training_curves, plot_grokking_analysis

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Parity Check

Binary sequence length 16, target is parity of the number of 1s.

Difficulty: ★☆☆☆☆

In [ ]:
X, y = generate_parity_data(5000, 16)
train_loader, test_loader = create_dataloaders(X, y, batch_size=64)

model = CSP(hidden_dim=64, output_dim=2, num_layers=3).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

losses, accs, grad_norms = train_model(
    model, train_loader, test_loader,
    epochs=100, lr=0.001, device=device,
    loss_fn=torch.nn.CrossEntropyLoss()
)

plot_training_curves(losses, accs, save_path='figures/parity_curves.png')
plot_grokking_analysis(accs, grad_norms, save_path='figures/parity_grokking.png')

print(f"Final Acc: {evaluate_model(model, test_loader, device):.4f}")
print(f"Final F1: {evaluate_model_f1(model, test_loader, device):.4f}")

## 2. Mod-3 Counting

Binary sequence length 16, target indicates whether the number of 1s is divisible by 3.

Difficulty: ★★☆☆☆

In [ ]:
X, y = generate_mod3_data(5000, 16)
train_loader, test_loader = create_dataloaders(X, y, batch_size=64)

model = CSP(hidden_dim=64, output_dim=2, num_layers=3).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

losses, accs, grad_norms = train_model(
    model, train_loader, test_loader,
    epochs=100, lr=0.001, device=device,
    loss_fn=torch.nn.CrossEntropyLoss()
)

plot_training_curves(losses, accs, save_path='figures/mod3_curves.png')
plot_grokking_analysis(accs, grad_norms, save_path='figures/mod3_grokking.png')

print(f"Final Acc: {evaluate_model(model, test_loader, device):.4f}")
print(f"Final F1: {evaluate_model_f1(model, test_loader, device):.4f}")

## 3. Parenthesis Matching

Binary sequence length 16 (0 for `(`, 1 for `)`), target indicates whether parentheses are balanced.

Difficulty: ★★★★★

In [ ]:
X, y = generate_parenthesis_data(10000, 16)
train_loader, test_loader = create_dataloaders(X, y, batch_size=64)

model = CSP(hidden_dim=128, output_dim=2, num_layers=3).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

from utils.train import focal_loss

losses, accs, grad_norms = train_model(
    model, train_loader, test_loader,
    epochs=300, lr=0.0001, device=device,
    loss_fn=focal_loss
)

plot_training_curves(losses, accs, save_path='figures/parenthesis_curves.png')
plot_grokking_analysis(accs, grad_norms, save_path='figures/parenthesis_grokking.png')

print(f"Final Acc: {evaluate_model(model, test_loader, device):.4f}")
print(f"Final F1: {evaluate_model_f1(model, test_loader, device):.4f}")

## 4. Summary

All three tasks should achieve near-perfect accuracy and F1 score, demonstrating the effectiveness of CSP on deterministic state tracking problems.

For more details, see the paper: [arXiv:2608.03425](https://arxiv.org/abs/2608.03425)